# 19.15 Looped 解題器怎樣成為獨立成品？


讀入有限事件、輸出正確答案，是 Looped 解題器的主要工作。為了先把共用深度的效果看清楚，第一版採明確文字語法。狀態追蹤沿用[14.15](https://birdhackor.github.io/tiny-perceptron-vlm/14.15.html)的兩盒世界：A與B各有0–7的初始值，事件依序把某盒設為指定值，或把兩盒的值交換，最後查詢其中一盒。輸入要照固定寫法，例如 `兩盒追蹤;A=2;B=6;事件=交換;查詢=B`：欄位之間用半形分號隔開，多個事件用半形逗號隔開，設值寫成 `A設5` 這種形式。這一題的正確答案是2；再交換一次（`事件=交換,交換`），B便回到6。因此重複交換要按順序處理，不能只記出現過哪些數字。

三數加法則給三個0–6整數，預測總和，寫成 `三數加法;x=1;y=2;z=3`，這一題的正確答案是6。兩類題都把預測整數與正確答案完全相同算答對；答案只有 0 到 18 這 19 種，所以最後輸出各答案類別的分數，不要求逐 token 生成一段說明。

tokenizer 把語法與數字轉成編號，embedding 表把它們變成向量。共用的 Transformer 區塊讀原輸入及目前狀態，反覆更新；最後由第一個位置的特徵經分類頭（程式裡的 `answer`）給 19 個答案類別分數。輸入順序仍然重要：「先設成5，再設成1」的答案是1，倒過來是5；位置資訊與改順序測試能幫我們檢查模型是否只數過哪些數字。

本版 Looped 解題器有170,003個參數，從seed7421的隨機權重開始，在NVIDIA L4完成1000次更新，依決策資料選出的也是第1000步。產品配方把同一題的1–5輪答案CE（交叉熵）取平均，讓每個部署深度都收到直接監督；問題的正確答案沒有隨輪數改變，模型需要學的是在不同計算預算下接近同一答案。

下面用隨機初值建立一個和成品同結構的 Looped 解題器，核對參數數與每一輪輸出的形狀；它沒有訓練過，不是成品，印出的 CE 不代表答題能力。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn

from tiny_perceptron.lesson_products.model import Block, Config, LoopedSolver, Tokenizer

torch.manual_seed(0)
tokenizer = Tokenizer()
model = LoopedSolver(Config(vocab_size=len(tokenizer)))
print("全部參數", sum(p.numel() for p in model.parameters()))
state = "兩盒追蹤;A=2;B=6;事件=交換;查詢=B"
logits = model(torch.tensor([tokenizer.encode(state)]), loops=5, all_depths=True)
print(state, "→ 各輪答案分數的形狀", tuple(logits.shape))
losses = nn.functional.cross_entropy(logits[0], torch.tensor([2] * 5), reduction="none")
print("1–5 輪各自的 CE", [round(loss, 3) for loss in losses.tolist()], "平均", round(losses.mean().item(), 3))

`Config(vocab_size=len(tokenizer))` 只指定字表大小，特徵寬度 128（`width`）、FFN 中間寬度 256（`ff`）等其餘設定都用預設值，和成品相同，所以全部參數印出 170,003；匯入的 `Block` 這段用不到，是下一段原始碼要用的。`all_depths=True` 讓模型交出每一輪的答案分數，形狀 `(1, 5, 19)` 依序是 1 題、5 輪、19 個答案類別。`nn.functional.cross_entropy(…, reduction="none")` 把 5 輪的分數各自和正確答案 2 比較（`[2] * 5` 是每輪一個 2），回傳 5 個 CE 而不先平均；它們的平均，就是產品配方替這一題算的訓練代價，因為五輪的權重都是 1。


In [ ]:
# 節錄自 tiny_perceptron/lesson_products/model.py：LoopedSolver
class LoopedSolver(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embedding = nn.Embedding(config.vocab_size, config.width, padding_idx=0)
        self.position = nn.Embedding(config.max_length, config.width)
        self.shared = Block(config)
        self.norm = nn.LayerNorm(config.width)
        self.answer = nn.Linear(config.width, 19)

    def forward(self, tokens, loops=3, all_depths=False, diagnostics=False):
        if type(loops) is not int or not 1 <= loops <= 6:
            raise ValueError("公開loop範圍是1–6")
        if tokens.shape[1] > self.config.max_length:
            raise ValueError("輸入超過max_length")
        original = self.embedding(tokens) + self.position(torch.arange(tokens.shape[1], device=tokens.device))
        hidden = original
        logits, norms = [], []
        for _ in range(loops):
            hidden = self.shared(hidden + self.config.input_injection * original, tokens != 0)
            logits.append(self.answer(self.norm(hidden[:, 0])))
            if diagnostics:
                norms.append(float(hidden.detach().float()[tokens != 0].square().mean().sqrt().item()))
        result = torch.stack(logits, dim=1) if all_depths else logits[-1]
        return (result, norms) if diagnostics else result

`__init__` 只建立**一個**區塊 `self.shared`，每一輪都用它；這個 `Block` 是 `lesson_products` 自己的版本，不是[4.5](https://birdhackor.github.io/tiny-perceptron-vlm/4.5.html)那一個：它的注意力不限於前文，每個位置都讀得到整段輸入；呼叫時傳入的 `tokens != 0` 是 `valid` 遮罩，讓注意力不讀補齊的 PAD 位置。`forward` 先算 `original`，也就是[14.12](https://birdhackor.github.io/tiny-perceptron-vlm/14.12.html)的 e（embedding 表查出的向量加上位置表 `position` 的向量）；迴圈每一輪都把 `hidden + self.config.input_injection * original` 交給同一個 `self.shared`，`input_injection` 預設 0.5，所以每輪都再讀一次原輸入。每輪結束都取第一個位置的特徵 `hidden[:, 0]`，經 `norm` 與 `answer` 得到 19 個答案類別分數；這個位置放的是 tokenizer 固定加在開頭的編號 1，不是題目的第一個字，它靠注意力讀到整題。`all_depths=True` 時 `torch.stack` 把各輪分數疊成上面的 `(1, 5, 19)`，否則只回傳最後一輪；`diagnostics` 另外記錄每輪特徵的均方根，本節用不到。

[14.15](https://birdhackor.github.io/tiny-perceptron-vlm/14.15.html)另做固定3輪與抽1／3／5輪的末端監督對照，資料、初始化與實際更新次數配對。那是檢查未直接監督的終止深度的機制實驗，不混用這份產品。完整最佳化器狀態、隨機狀態與成對檢查點另存，且本次L4實際核對了恢復後下一步一致；只下載推論權重不包含這些續訓狀態。

交付不是只有權重檔。設定要足以重建網路；tokenizer要讓同一文字得到同一編號；推論指令要能選輪數、在CPU載入、回傳答案及實際執行設定；manifest把這些檔綁到固定版本。訓練指令另保存資料、最佳化器、隨機狀態及選權重依據，讀者才分得清繼續訓練與只作推論。

預先訂定的主要門檻是五輪狀態追蹤至少90%。凍結後的測試得到2928/2956，約99.05%，通過主要門檻；加法只有2/42，約4.76%，未達85%的次要目標。不能因主要門檻通過，就把 Looped 解題器稱成可靠的三數加法器。

平均數也會藏住子群。**重複交換**是事件裡至少交換兩次的題目，2,956 道追蹤測試題中有 42 題，只答對 32 題（這個 42 和加法的 42 題只是剛好相同）；把盒子名稱 A、B 換成 P、Q 後只對 1486/2956；把事件長度拉到 5–6 時只對 1245/3200。這版通過的是原A/B命名、原句型與1–4次事件的有限測試題，仍有上述子群失敗；長事件與改名只作壓力測試。這些結果都保存在實驗紀錄 [final-report-v1.json](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/lesson-products/results/final-report-v1.json)，它是機器可讀的完整結果（看 `b` 欄的 `"5"`，以及 `shifts.entity_shift`、`shifts.long_shift`）。

本版low／medium／high事前固定為1／3／5輪，代表允許的工作設定，不是三個品質保證。即使1–5輪都直接受監督，一輪追蹤仍只有1598/2956，約54.06%；四輪2930/2956還略高於五輪2928/2956。實驗紀錄也測了第 6 輪（2925/2956）：產品配方只監督 1–5 輪，第 6 輪用來看超出訓練深度時的表現，不在部署的檔位裡。這使我們能直接看見：多深度訓練促進各輪答題，但不保證每個檔位都達90%，也不保證越多輪越好。[19.18](https://birdhackor.github.io/tiny-perceptron-vlm/19.18.html)會把相同測試題的品質與成本一起列出。

練習：把 19.13 例子裡的「事件=A設5,A設1」刪掉最後一次覆寫，變成「事件=A設5」，答案應如何變？先用世界規則求正確答案（5），再看模型輸出。這種只改關鍵事件的小變化，比只挑一個漂亮成功例更能查出是否使用材料。
